In [1]:
# 1. Instalação e Inicialização do PySpark
!pip install pyspark -q

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("Sistematizacao_KDD_Diabetes") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()
print("Sessão Spark iniciada com sucesso!")


Sessão Spark iniciada com sucesso!


In [2]:
# 2. Download do dataset diretamente para o ambiente Colab
!pip install kagglehub -q
import kagglehub, shutil

path = kagglehub.dataset_download('alexteboul/diabetes-health-indicators-dataset')
shutil.copy(f'{path}/diabetes_binary_health_indicators_BRFSS2015.csv', './diabetes_binary_health_indicators_BRFSS2015.csv')
print("Dataset baixado com sucesso!")


Using Colab cache for faster access to the 'diabetes-health-indicators-dataset' dataset.
Dataset baixado com sucesso!


In [3]:
# 3. Leitura e limpeza (Remoção de duplicadas/nulos)
df = spark.read.csv(
    "diabetes_binary_health_indicators_BRFSS2015.csv",
    header=True,
    inferSchema=True
)

total_bruto = df.count()
df_limpo = df.dropDuplicates() .na.drop()
total_limpo = df_limpo.count()

print(f"Total bruto: {total_bruto}")
print(f"Total limpo: {total_limpo} (Duplicadas removidas): {total_bruto - total_limpo}")

# 4. Engenharia de atributos e criação de view temporária para SQL
df_transformado = df_limpo.withColumn("Risco_Cardiovascular", F.col("HighBP") + F.col("HighChol"))
df_transformado.createOrReplaceTempView("tabela_diabetes")
print("Tabela SQL 'tabela_diabetes' e variável 'df_transformado' prontas!")

Total bruto: 253680
Total limpo: 229474 (Duplicadas removidas): 24206
Tabela SQL 'tabela_diabetes' e variável 'df_transformado' prontas!


In [4]:
# Pergunta 1: Prevalência geral de Diabetes na amostra
print("=== 1. Proporção de Doença na Amostra ===")
spark.sql("""
 SELECT
    Diabetes_binary AS Diagnostico,
    COUNT(*) AS Total,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS Percentual
  FROM tabela_diabetes
  GROUP BY Diabetes_binary
""").show()

# Pergunta 2: IMC Médio por Diagnóstico
print("=== 2. Média e Dispersão do IMC ===")
spark.sql ("""
 SELECT
    Diabetes_binary AS Diagnostico,
    ROUND(AVG(BMI), 2) AS Media_IMC,
    ROUND(STDDEV(BMI), 2) AS Desvio_IMC
  FROM tabela_diabetes
  GROUP BY Diabetes_binary
""").show()

# Pergunta 3: Risco Cardiovascular Combinado (Hipertensão + Colesterol)
print("=== 3. Prevalência por Risco Cardiovascular (0=Nenhum, 1= Um deles, 2= Ambos) ===")
spark.sql("""
 SELECT
    Risco_Cardiovascular,
    COUNT(*) AS Total_Populacao,
    SUM(Diabetes_binary) AS Total_Diabeticos,
    ROUND(SUM(Diabetes_binary) * 100.0 / COUNT(*), 2) AS Taxa_Diabetes_Pct
  FROM tabela_diabetes
  GROUP BY Risco_Cardiovascular
  ORDER BY Risco_Cardiovascular
""") .show()

# Pergunta 4: Prevalência de Diabetes por Faixa Etária
print("=== 4. Prevalência de Diabetes Idade (1=Jovem até 13=Idoso) ===")
spark.sql("""
 SELECT
    AGE AS Faixa_Idade,
    COUNT(*) AS Total,
    ROUND(SUM(Diabetes_binary) * 100.0 / COUNT(*), 2) AS Taxa_Diabetes_Pct
  FROM tabela_diabetes
  GROUP BY Age
  ORDER BY Age
""").show(14)

# Pergunta 5: Efeito de Hábitos Saudáveis
print("=== 5. Hábitos Saudáveis x Diabetes ===")
spark.sql("""
 SELECT
    PhysActivity AS Atividade_Fisica,
    Veggies AS Come_Vegetais,
    COUNT(*) AS Total,
    ROUND(SUM(Diabetes_binary) * 100.0 / COUNT(*), 2) AS Taxa_Diabetes_Pct
  FROM tabela_diabetes
  GROUP BY PhysActivity, Veggies
  ORDER BY Taxa_Diabetes_Pct DESC
""").show()

=== 1. Proporção de Doença na Amostra ===
+-----------+------+----------+
|Diagnostico| Total|Percentual|
+-----------+------+----------+
|        0.0|194377|     84.71|
|        1.0| 35097|     15.29|
+-----------+------+----------+

=== 2. Média e Dispersão do IMC ===
+-----------+---------+----------+
|Diagnostico|Media_IMC|Desvio_IMC|
+-----------+---------+----------+
|        0.0|     28.1|       6.5|
|        1.0|    31.96|      7.38|
+-----------+---------+----------+

=== 3. Prevalência por Risco Cardiovascular (0=Nenhum, 1= Um deles, 2= Ambos) ===
+--------------------+---------------+----------------+-----------------+
|Risco_Cardiovascular|Total_Populacao|Total_Diabeticos|Taxa_Diabetes_Pct|
+--------------------+---------------+----------------+-----------------+
|                 0.0|          86026|          4246.0|             4.94|
|                 1.0|          81291|         11801.0|            14.52|
|                 2.0|          62157|         19050.0|           

In [5]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml import Pipeline

# 5. Selecionar preditoras clínicas e comportamentais
preditoras = [
    'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker',
    'Stroke', 'HeartDiseaseorAttack','PhysActivity', 'Fruits', 'Veggies',
    'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth',
    'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education','Income'
]

assembler = VectorAssembler(inputCols=preditoras, outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=True)

# 6. Divisão treino/teste (75/25)
treino, teste = df_transformado.randomSplit([0.75, 0.25], seed=42)

# 7. Modelo 1: Regressão Logística (Linear)
lr = LogisticRegression(featuresCol="features", labelCol="Diabetes_binary", maxIter=20)
pipeline_lr = Pipeline(stages=[assembler, scaler, lr])
modelo_lr = pipeline_lr.fit(treino)
predicoes_lr = modelo_lr.transform(teste)

# 8. Modelo 2: Random Forest (Esemble)
rf = RandomForestClassifier(featuresCol="features_raw", labelCol="Diabetes_binary", numTrees=50, maxDepth=8, seed=42)
pipeline_rf = Pipeline(stages=[assembler, rf])
modelo_rf = pipeline_rf.fit(treino)
predicoes_rf = modelo_rf.transform(teste)

# 9. Avaliação comparativa das métricas
eval_auc = BinaryClassificationEvaluator(labelCol="Diabetes_binary", metricName="areaUnderROC")
eval_acc = MulticlassClassificationEvaluator(labelCol="Diabetes_binary", metricName="accuracy")
eval_f1 = MulticlassClassificationEvaluator(labelCol="Diabetes_binary", metricName="f1")

auc_lr = eval_auc.evaluate(predicoes_lr)
acc_lr = eval_acc.evaluate(predicoes_lr)
f1_lr = eval_f1.evaluate(predicoes_lr)

auc_rf = eval_auc.evaluate(predicoes_rf)
acc_rf = eval_acc.evaluate(predicoes_rf)
f1_rf = eval_f1.evaluate(predicoes_rf)

print("=" * 60)
print("RESULTADOS COMPARATIVOS - MODELAGEM PREDITIVA (MLlib)")
print("=" * 60)
print(f"Regressão Logística -> (AUC-ROC): {auc_lr:.4f} | Acurácia: {acc_lr*100:.2f}% | F1: {f1_lr:.4f}")
print(f"Random Forest       -> (AUC-ROC): {auc_rf:.4f} | Acurácia: {acc_rf*100:.2f}% | F1: {f1_rf:.4f}")
print("=" * 60)

RESULTADOS COMPARATIVOS - MODELAGEM PREDITIVA (MLlib)
Regressão Logística -> (AUC-ROC): 0.8059 | Acurácia: 85.04% | F1: 0.8114
Random Forest       -> (AUC-ROC): 0.8002 | Acurácia: 85.05% | F1: 0.7956


In [6]:
# 10. Modelagem preditiva / não supervisionada (K-MEANS)
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# 10.1 Selecionar atributos clínicos er de estilo de vida para segmentaçaõ
features_cluster = ['BMI', 'GenHlth', 'PhysHlth', 'MentHlth', 'Age']
assembler_k = VectorAssembler(inputCols=features_cluster, outputCol="feat_raw")
scaler_k = StandardScaler(inputCol="feat_raw", outputCol="features_cluster", withStd=True, withMean=True)

dados_k = scaler_k.fit(assembler_k.transform(df_transformado)).transform(assembler_k.transform(df_transformado))

# 10.2 Avaliação de Silhueta para justificar a escolha de k=3
evaluator_sil = ClusteringEvaluator(featuresCol="features_cluster", metricName="silhouette", distanceMeasure="squaredEuclidean")

print("--- Avaliação do Coeficiente de Silhueta ---")
for k in [2,3,4]:
  km = KMeans(featuresCol="features_cluster", k=k, seed=42)
  mod = km.fit(dados_k)
  sil = evaluator_sil.evaluate(mod.transform(dados_k))
  print(f"k = {k} -> Silhoette Score = {sil:.4f}")

# 10.3 Treinar modelo final com k=3
kmeans_final = KMeans(featuresCol="features_cluster", k=3, seed=42, predictionCol="cluster")
modelo_km = kmeans_final.fit(dados_k)
df_clusters = modelo_km.transform(dados_k)

#10.4 Tabela de Perfil dos Agrupamentos em Linguagem de Negócio
df_clusters.createOrReplaceTempView("tabela_clusters")

perfil = spark.sql("""
 SELECT
    cluster,
    COUNT(*) AS Total_Individuos,
    ROUND(AVG(BMI), 1) AS Media_IMC,
    ROUND(AVG(GenHlth), 1) AS Media_Autavaliacao_Saude,
    ROUND(AVG(PhysHlth), 1) AS Media_Dias_Saude_Ruim,
    ROUND(AVG(Age), 1) AS Media_Faixa_Etaria,
    ROUND(SUM(Diabetes_binary) * 100.0 / COUNT(*), 2) AS Prevalencia_Diabetes_Pct
  FROM tabela_clusters
  GROUP BY cluster
  ORDER BY Prevalencia_Diabetes_Pct ASC
""")

print("\n=== PERFIL DOS GRUPOS (CLUSTERS) IDETIFICADOS ===")
perfil.show()

--- Avaliação do Coeficiente de Silhueta ---
k = 2 -> Silhoette Score = 0.6054
k = 3 -> Silhoette Score = 0.3633
k = 4 -> Silhoette Score = 0.3916

=== PERFIL DOS GRUPOS (CLUSTERS) IDETIFICADOS ===
+-------+----------------+---------+------------------------+---------------------+------------------+------------------------+
|cluster|Total_Individuos|Media_IMC|Media_Autavaliacao_Saude|Media_Dias_Saude_Ruim|Media_Faixa_Etaria|Prevalencia_Diabetes_Pct|
+-------+----------------+---------+------------------------+---------------------+------------------+------------------------+
|      1|           76306|     28.3|                     2.1|                  1.2|               4.8|                    4.61|
|      0|          117040|     28.3|                     2.5|                  1.5|              10.1|                   17.93|
|      2|           36128|     31.0|                     3.9|                 22.2|               8.6|                   29.34|
+-------+----------------+--------